In [ ]:
import gymnasium as gym
import numpy as np
import subprocess
import os
import wave
from gymnasium.wrappers import RecordVideo
from gymnasium.wrappers import ResizeObservation
from stable_baselines3.common.atari_wrappers import WarpFrame, MaxAndSkipEnv
from sdlarch_rl import make
import pygame
import cv2
import time

import numpy as np

os.makedirs("videos", exist_ok=True)


video_filename = "videos/video-episode-0.mp4"
audio_filename = "videos/game_audio.wav"

audio_data = b""
arate = None
framerate = None

data = {
    "audio_data": audio_data,
    "arate": arate,
    "framerate": framerate,
}

frames = []
# env = make("GranTurismo3-Ps2", env_variables=[{"pcsx2_upscale_multiplier", "3x Native (~1080p)"}])
env = make("SuperStreetFighterIV-3DS")
#env = make("NewSuperMarioBros-Wii", env_variables=[{"dolphin_efb_scale", "x2 (1280 x 1056)"}])
# env = make("MarioDash-GC", env_variables=[{"dolphin_efb_scale", "x2 (1280 x 1056)"}])
# env = make("VirtuaTennis-DC")
# env = make("MarvelVsCapcom2-DC")
# env = make("SuperMario64-N64")
# env = make("NewSuperMarioBros-Wii")
# env = MaxAndSkipEnv(env, skip=4)
# env = make("GTASanAndreas-Ps2", env_variables=[{"pcsx2_upscale_multiplier", "3x Native (~1080p)"}])
# env = make("CrazyTaxi-DC")
# env = make("SuperSmashBrosBrawl-Wii")
# env = make("GodOfWar-PSP")
# env = make("YoshiIsland-NDS")

# env = RecordVideo(
#     env,
#     video_folder="videos/",
#     episode_trigger=lambda x: x == 0,
#     name_prefix="video"
# )


render_mode = "rgb_array"
# render_mode = "human"


obs, info = env.reset()

print(obs.shape)

done = False
count = 0

pygame.init()

height, width, _ = obs.shape
SCREEN_WIDTH = 1920
SCREEN_HEIGHT = int(SCREEN_WIDTH * (height / width))

window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
clock = pygame.time.Clock()

data['arate'] = env.unwrapped.em.get_audio_rate()
data['framerate'] = env.unwrapped.em.get_frame_rate()

print("arate: ", data['arate'])
print("framerate: ", data['framerate'])

frame_time = 1.0 / data['framerate']
last_time = time.time()

# Wii
INVERT_AXIS = False

step_count = 0

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            done = True
            
    keys = pygame.key.get_pressed()

    env.render()

    action = np.zeros(16, dtype=np.uint8)

    # pad
    if keys[pygame.K_UP]:
        if INVERT_AXIS:
            action[7] = 1
        else:
            action[4] = 1
    if keys[pygame.K_DOWN]:
        if INVERT_AXIS:
            action[6] = 1
        else:
            action[5] = 1
    if keys[pygame.K_LEFT]:
        if INVERT_AXIS:
            action[4] = 1
        else:
            action[6] = 1
    if keys[pygame.K_RIGHT]:
        if INVERT_AXIS:
            action[5] = 1
        else:
            action[7] = 1

    # buttons
    if keys[pygame.K_x]:
        action[0] = 1
    if keys[pygame.K_c]:
        action[1] = 1
    if keys[pygame.K_y]:
        action[8] = 1
    if keys[pygame.K_z]:
        action[9] = 1
    if keys[pygame.K_q]:
        action[10] = 1
    if keys[pygame.K_w]:
        action[11] = 1
    if keys[pygame.K_RETURN]:
        action[3] = 1
    # if keys[pygame.K_l]:
    #     action[10] = 1

    # clock.tick(60)

    img, rew, done, _, info = env.step(action)

    
    # print(info)
    # print(rew)
    sound = env.unwrapped.em.get_audio()
    #print(sound.shape)
    data['audio_data'] += sound.tobytes()

     # stop
    if keys[pygame.K_BACKSPACE]:
        print("== DONE ==")
        done = True

    # frame rate
    now = time.time()
    sleep_time = frame_time - (now - last_time)
    if sleep_time > 0:
        time.sleep(sleep_time)
    last_time = now

    
    img = cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT))

    if img is not None:
        frame = cv2.resize(img, (640, 480))
        frames.append(frame)

    surface = pygame.surfarray.make_surface(np.transpose(img, (1, 0, 2)))

    window.blit(surface, (0, 0))
    pygame.display.update()

    count += 1


    if done:
        break

env.close()
pygame.quit()

out = cv2.VideoWriter(video_filename, cv2.VideoWriter_fourcc(*"mp4v"), 60, (640, 480))
for f in frames:
    out.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
out.release()

print("arate: ", data['arate'])
print("framerate: ", data['framerate'])

# save audio
with wave.open(audio_filename, "wb") as wf:
    wf.setnchannels(2)  # Mono
    wf.setsampwidth(2)  # 16-bit PCM
    wf.setframerate(data['arate'])  # Audio rate
    wf.writeframes(data['audio_data'])

video_prefix = "video.mp4"


ffmpeg_cmd = [
    "ffmpeg", "-y",
    "-r", str(data['framerate']),
    "-i", video_filename,
    "-i", audio_filename,
    # "-vf", "scale=1280:720", # hd resolution
    "-vf", f"scale={SCREEN_WIDTH}:{SCREEN_HEIGHT},setdar=16:9,setsar=1", # HD
    # "-vf", "scale=854:480,setdar=16:9,setsar=1", # 16x9
    "-c:v", "libx264",
    # "-preset", "veryslow",
     "-crf", "10", # quality 10 ~ 50 (10 is better)
    "-c:a", "aac",
    "-b:a", "128k",
    "-ac","2",
    "-map", "0:v:0",       
    "-map", "1:a:0",
    "-strict", "experimental",
    "-shortest",
    "./videos/" + video_prefix
]

subprocess.run(ffmpeg_cmd)

print("✅ Finished vídeos")

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


statename is None setting to default state
State file not found: D:\projects\sdlarch-rl\sdlarch_rl\roms\SuperStreetFighterIV-3DS\default.state. Starting without initial state.
(240, 400, 3)
arate:  32728.0
framerate:  60.0
